![Platform+](images/banner.png)

<div style="max-width:150px; float:right;">

![The Roastery Coffee Co.](images/roastery-logo.png)

</div>

# Lab 10: Capstone — Roastery Intelligence
## Govern the investigation — was it efficient, and was it safe?

> **Persona track: Platform Admin** — same mission, your angle.

:point_right: **In this lesson, you'll audit how the revenue-dip investigation was actually run — then close the gaps you find.**

The scenario: Another team investigated a revenue dip the VP of Operations flagged at Roastery Mission and stood up a pipeline to keep its data fresh. You didn't run that work — your job isn't the analysis. It's making sure it was done responsibly, and that it stays that way.

We'll cover:
1. Audit the investigation for cost and access
2. Consume a handoff from the data analyst team — spot the PII leak hiding in a summary table
3. Build the governance layer: masking policy + resource monitor
4. Govern the load path — stage, load, and grant in one motion
5. Govern marketplace access — curated projection, not wholesale resharing
6. Audit history & recover — Time Travel as the admin safety net
7. Verify your work and reflect on what you'd flag in a real review

📚 **By the end you'll be able to:**
- Turn a governance goal into a self-authored CoCo prompt, not a copy-pasted one
- Spot ungoverned data hiding in a "clean-looking" summary table
- Apply targeted remediation rather than a blanket policy
- Prove — with `QUERY_HISTORY` and `SHOW GRANTS` — that the fix actually worked

**Time:** ~45 min · **Lab interface:** Cortex Code (CoCo) is your tutor — ask it to explain or fix anything.
*Source data lives in the shared, read-only `PLATPLUS_DB`; anything you create lives in your own database (named after your username).*

---
👉 **Connecting to your compute service:**
1. In the top bar, if no service is connected, click to connect and select your service.
2. Set your **role and warehouse** using the picker at the top-right — or just run the setup cell below, which sets them for you in SQL. (SQL cells run on the warehouse.)
3. Once the status shows **Connected** (green), run a cell with **Shift+Enter** (run + advance) or **Cmd/Ctrl+Enter** (run this cell only); the **▶** button works too.

> 📌 The service is a container that runs the notebook kernel and your **Python** cells. **SQL cells (and Snowpark pushdown) run on the query warehouse**, not the container. It stays running until it hits its idle timeout or you suspend it manually.
> 💡 **Best practice: Start a fresh chat for this lab.** Before you begin, open a fresh CoCo chat — click **New chat** (the **+** at the top of the panel). Each lab creates its own objects, so a fresh chat ensures CoCo isn't referencing tables or schemas from a prior lab that no longer exist. Shorter conversations also produce sharper answers and use fewer tokens.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
user = session.get_current_user().strip('"')
my_db = user + '_DB'
my_wh = user + '_WH'
my_role = user + '_RL'
my_analyst_role = user + '_ANALYST_RL'
print('Context ready. Signed in as ' + user)
print('Your workspace -> db: ' + my_db + ' | wh: ' + my_wh + ' | role: ' + my_role)


> ## Set your session context
> Every query runs under a role (what you're allowed to do) and a warehouse (compute). The cell below switches to your own role, creates and uses your own warehouse, and creates a `RAW` schema for what you'll build in this lab.

In [ ]:
USE ROLE {{my_role}};
CREATE WAREHOUSE IF NOT EXISTS {{my_wh}}
  WAREHOUSE_SIZE = XSMALL AUTO_SUSPEND = 60 AUTO_RESUME = TRUE INITIALLY_SUSPENDED = TRUE;
USE WAREHOUSE {{my_wh}};
CREATE DATABASE IF NOT EXISTS {{my_db}};
CREATE SCHEMA IF NOT EXISTS {{my_db}}.RAW;


> :mortar_board: **This is a capstone — not a tutorial.** Every section calls back to a skill from an earlier lab. Most cells are pre-filled (run them and observe); one section asks you to write SQL with CoCo's help. Each section gives you **three paths to the answer (your choice)** — you can try all three if you wish, but you only need one:
>
> 1. **Write it yourself** in the empty cells (the real challenge — prove you learned it)
> 2. **Reveal the 🧩 Stuck? Reference solution** and paste it in (see the answer directly)
> 3. **Use the 🤖 CoCo assist prompt** to have CoCo generate the code for you (practice the AI-assisted workflow)
>
> All three paths produce the same result going into the same empty cells. Pick whichever level of help you need — there's no penalty for using the hints.

> :rotating_light: **IMPORTANT — CoCo vs notebook SQL cells:** As you go through this lab and use CoCo, be aware that curly braces `{}` in SQL can sometimes conflict with CoCo's internal SQL execution. If CoCo reports an error but the SQL looks correct, just copy it into a notebook SQL cell and run it there — it will work fine. This is a difference in how CoCo executes SQL internally vs how notebook cells send it directly to Snowflake.

---
## 1 · Investigate — audit the investigation itself (Labs 01, 02, 04 — Navigate + Verify + Cost)

You're not chasing the dip's cause — the Data Analyst already did that. Your job is to check *how* that investigation was run: was it efficient, and was it safe?

**Why this matters (platform admin perspective):**
- **Efficiency and safety are your metrics, not correctness** — Someone else validates the analysis. You validate that it was run responsibly: reasonable compute, no unnecessary PII exposure.
- **A clean answer can still be an expensive or risky one** — A correct SQL result says nothing about whether it scanned way more data than needed, or joined to a table it had no business touching.
- **Audits should be repeatable** — The two checks below (query cost, grant exposure) are the same two questions you'd run after *any* investigation, not just this one.

**The audit workflow:**
1. **Efficient?** Pull recent query activity against the tables the investigation touched — bytes scanned, warehouse, execution time.
2. **Secure?** Check whether the role that ran the investigation ever had a path to `CUSTOMERS.FULL_NAME` — real customer names.

**First, know what you're governing.** Before auditing *how* the investigation ran, map *what* exists in the schemas it touched. Lab 01's habit: navigate the objects before you reason about them.

> 📌 **Recall Lab 01:** `INFORMATION_SCHEMA` and `SHOW` let you inventory a schema without querying every table.

In [ ]:
-- Map the objects in the schemas the investigation touched
SELECT table_schema, table_type, COUNT(*) AS object_count,
       LISTAGG(table_name, ', ') WITHIN GROUP (ORDER BY table_name) AS objects
FROM PLATPLUS_DB.INFORMATION_SCHEMA.TABLES
WHERE table_schema IN ('ROASTERY_SALES', 'ROASTERY_INTELLIGENCE')
GROUP BY table_schema, table_type
ORDER BY table_schema, table_type;


> 🔎 **Review.** `ROASTERY_SALES` holds the 6 raw base tables (+ a `SALES_BY_STORE` view); `ROASTERY_INTELLIGENCE` holds 3 governed views (`V_FEEDBACK_SAFE`, `V_PRODUCTS_ENRICHED`, `FEEDBACK_SCORED_SUMMARY`). This is your governance surface: raw tables on one side, the derived/published objects analysts actually consume on the other. The leak you'll find later lives in that second group — which is exactly why an admin inventories *derived* objects, not just raw ones.

In [ ]:
-- Recent activity against the tables the dip investigation touched
SELECT LEFT(query_text, 90)                    AS query_preview,
       warehouse_name,
       ROUND(execution_time / 1000.0, 1)        AS secs,
       ROUND(bytes_scanned / 1024 / 1024, 1)    AS mb_scanned,
       ROUND(credits_used_cloud_services, 5)    AS cloud_svc_credits
FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY
WHERE start_time >= DATEADD(day, -7, CURRENT_TIMESTAMP())
  AND (query_text ILIKE '%ROASTERY_SALES.ORDERS%' OR query_text ILIKE '%ROASTERY_SALES.STORES%')
ORDER BY start_time DESC
LIMIT 10;


> 🔎 **How to read this.** `MB_SCANNED` and `SECS` are your efficiency signals — a dip analysis over a few thousand rows shouldn't be scanning gigabytes or running for minutes on an XSMALL. An empty result just means no matching queries ran in the last 7 days in this account — that's fine, the pattern is what matters.
>
> ⏱️ **Latency note:** `QUERY_HISTORY` lags up to ~45 minutes. Queries from earlier in *this* session may not show up yet.

In [ ]:
-- Does {{my_role}} — the role that ran the dip investigation — have any path to CUSTOMERS?
SHOW GRANTS TO ROLE {{my_role}};
SELECT * FROM TABLE(RESULT_SCAN(LAST_QUERY_ID())) WHERE "name" ILIKE '%CUSTOMERS%';


> 🔎 **Read the second result.** You may see a single row showing a `REFERENCES` privilege on `CUSTOMERS` — that's expected and harmless. `REFERENCES` is a metadata-only grant (used for foreign-key validation); it does **not** allow reading data. No `SELECT` grant means `{{my_role}}` cannot access the rows in `CUSTOMERS` — real customer names were never reachable through the raw table for this investigation.
>
> **But don't stop here.** A raw table being locked down doesn't mean PII can't leak somewhere else — a *derived* table, a summary, a rollup someone built on top of the raw data. That's exactly what you're about to check.

**Don't trust one signal.** The grant check above said `{{my_role}}` has no grant on `CUSTOMERS`. Lab 02's habit: corroborate a finding with a second, independent check. Grant *metadata* says one thing — does *enforcement* actually agree? Try to read the raw table directly.

> 📌 **Recall Lab 02:** verify — a second corroborating query beats trusting a single result.

In [ ]:
-- Corroborate the grant finding with actual enforcement — this is EXPECTED to fail
SELECT COUNT(*) FROM PLATPLUS_DB.ROASTERY_SALES.CUSTOMERS;


> 🔒 **Access denied — and that's the corroboration.** The grant metadata said "no path"; enforcement agrees by refusing the read outright. Two independent signals pointing the same way is a finding you can sign off on. One signal alone — the grant view *or* the failed query — could be misread; together they're conclusive. (The error is expected — it's the proof, not a mistake.)

---
## 2 · Consume 📦 — a governed handoff (Labs 03, 05 — RBAC + AI)

In a real workplace, a Data Analyst team would publish summary tables for other teams to consume. In this lab, that's already been done for you — `PLATPLUS_DB.ROASTERY_INTELLIGENCE.FEEDBACK_SCORED_SUMMARY` is a pre-provisioned per-store rollup of feedback sentiment, meant to sit alongside the dip analysis. It's a summary table, not raw feedback, so it should already be clean of anything sensitive. You only have `SELECT` on it — you're consuming, not building this one.

> 📌 **You don't need to run any other lab first.** This table is already provisioned in the shared `PLATPLUS_DB` database.

**Why this matters (platform admin perspective):** you don't just govern raw tables. Every derived object downstream of raw data needs the same scrutiny — summaries and rollups are exactly where PII quietly survives an aggregation that was supposed to remove it.

In [ ]:
SELECT * FROM PLATPLUS_DB.ROASTERY_INTELLIGENCE.FEEDBACK_SCORED_SUMMARY
ORDER BY avg_score;


> 🔍 **Look closely before you sign off on this.** Every column except one is a legitimate aggregate — `AVG_SCORE`, `FEEDBACK_COUNT`, `MOST_RECENT_FEEDBACK_DATE`. But `MOST_RECENT_REVIEWER` is a **customer identifier**, sitting in what's supposed to be a rolled-up summary table. That's exactly the kind of "clean-looking" leak the audit in Section 1 wouldn't have caught — it's not the raw `CUSTOMERS` table, it's downstream of it.
>
> This is the gap you're about to close.

---
## 3 · Build — the governance layer (Labs 03, 04 — RBAC + Cost)

You can't fix the shared handoff object directly — you only have `SELECT` on `PLATPLUS_DB.ROASTERY_INTELLIGENCE`. So you'll copy it into your own database (where you're the owner and can govern it), then remediate.

**Step 1 — copy the handoff into your own database.**

In [ ]:
CREATE OR REPLACE TABLE {{my_db}}.RAW.FEEDBACK_SCORED_SUMMARY AS
SELECT * FROM PLATPLUS_DB.ROASTERY_INTELLIGENCE.FEEDBACK_SCORED_SUMMARY;


**Step 2 — apply targeted remediation, not a blanket policy.**

The fix isn't to drop the column or hide the whole table — it's a **masking policy** on just `MOST_RECENT_REVIEWER`, so `{{my_role}}` (you, the owner) still sees the real value while `{{my_analyst_role}}` (a lower-privileged role) sees it masked.

> 🤖 **CoCo assist** *(try the empty cell yourself first — if stuck, reveal the 🧩 solution below OR ask CoCo in your own words):*
>
> State the goal yourself in the CoCo panel. Your prompt should describe:
> - The table and column: `{{my_db}}.RAW.FEEDBACK_SCORED_SUMMARY`, column `MOST_RECENT_REVIEWER`
> - The policy logic: full value for `{{my_role}}`, masked (e.g. `***MASKED***`) for every other role
> - That the generated SQL should do **both** — create the masking policy **and** attach it to the column with `ALTER TABLE … SET MASKING POLICY` (two statements) — so running it applies the mask end-to-end
>
> Paste CoCo's SQL into the cell below and run it. If it errors, paste the error back to CoCo and iterate — that's the normal workflow, not a sign you did something wrong.

In [ ]:
-- YOUR TURN: paste CoCo's masking-policy SQL on the blank line below, then run it


**Paste into the `build_masking_paste_cell` cell (the YOUR TURN cell above):**

<details>
<summary>🧩 Stuck? Reference solution</summary>

Two statements: (1) create the policy with a **`CASE`** that reveals the value to your role and returns `***MASKED***` for everyone else, (2) attach it to the specific column with `ALTER TABLE … MODIFY COLUMN … SET MASKING POLICY`.

```sql
CREATE OR REPLACE MASKING POLICY {{my_db}}.RAW.MASK_REVIEWER_NAME AS (val STRING) RETURNS STRING ->
  CASE
    WHEN CURRENT_ROLE() = '{{my_role}}' THEN val
    ELSE '***MASKED***'
  END;

ALTER TABLE {{my_db}}.RAW.FEEDBACK_SCORED_SUMMARY
  MODIFY COLUMN MOST_RECENT_REVIEWER SET MASKING POLICY {{my_db}}.RAW.MASK_REVIEWER_NAME;
```

</details>

In [ ]:
-- Confirm the policy is attached — no role-switching needed to check this
SELECT POLICY_NAME, POLICY_KIND, REF_COLUMN_NAME, POLICY_STATUS
FROM TABLE({{my_db}}.INFORMATION_SCHEMA.POLICY_REFERENCES(
  REF_ENTITY_NAME => '{{my_db}}.RAW.FEEDBACK_SCORED_SUMMARY',
  REF_ENTITY_DOMAIN => 'TABLE'
));


> 🔎 **Review.** You should see one row: `MASK_REVIEWER_NAME`, kind `MASKING_POLICY`, attached to `MOST_RECENT_REVIEWER`, status `ACTIVE`. This confirms the policy is *attached* — but attaching isn't proving. **Section 7** closes the loop by demonstrating it actually masks under the analyst role. (We separate _apply_ from _prove_ on purpose: applying a policy and verifying it holds under a different identity are two distinct admin skills.)

**Step 3 — a resource monitor, so the fleet stays affordable too.**

Governance isn't only about who sees what — it's also about spend. Reuse the same helper procs from Lab 4: `CREATE RESOURCE MONITOR` and attaching one to a warehouse both require `ACCOUNTADMIN`, so these are wrapped as owner's-rights stored procs you can call directly.

The sequence below runs three procs:
1. **`GRANT_MONITOR_USAGE`** — grants your role `MONITOR` on the resource monitor so `SHOW RESOURCE MONITORS` actually returns it (without this, ACCOUNTADMIN-owned monitors are invisible to you)
2. **`CREATE_DEMO_RESOURCE_MONITOR`** — creates a resource monitor named `<USER>_RM` with a 5-credit daily quota and a 100% suspend trigger
3. **`ATTACH_DEMO_RESOURCE_MONITOR`** — attaches that monitor to your warehouse (`{{my_wh}}`), so the warehouse suspends automatically if it crosses the quota

In [ ]:
CALL PLATPLUS_DB.PUBLIC.GRANT_MONITOR_USAGE(CURRENT_USER());


In [ ]:
CALL PLATPLUS_DB.PUBLIC.CREATE_DEMO_RESOURCE_MONITOR(CURRENT_USER());
CALL PLATPLUS_DB.PUBLIC.ATTACH_DEMO_RESOURCE_MONITOR(CURRENT_USER());
SHOW WAREHOUSES LIKE '{{my_wh}}';


> 🔎 **Review.** The `resource_monitor` column on `{{my_wh}}` now shows `<YOUR_USERNAME>_RM` — a 5-credit quota that suspends the warehouse at 100%. Same pattern as Lab 4: creation and attachment are `ACCOUNTADMIN`-only actions, wrapped in a proc so you don't need that role.

---
## 4 · Govern the load path (Lab 07 — Getting Data In)

You found a leak, you fixed it, and you capped the spend. But an audit that only lives in your head isn't governance — it's tribal knowledge. The next step: **record what you found.** An access-review roster documents which roles had access to which objects, what was approved, and what was denied. It's the paper trail that lets someone *else* verify your audit six months from now.

And here's the admin's instinct: you don't just write the file — you control where it lands and who can read it, *at the moment of loading*. Governance on the load path means the data is governed from the instant it exists, not as an afterthought.

> 📌 **Recall Lab 07:** the load pattern is always **stage → file format → COPY INTO**; `SKIP_HEADER=1` because the CSV has a header row.

In [ ]:
-- Stage + file format for the access-review CSV
CREATE STAGE IF NOT EXISTS {{my_db}}.RAW.GOV_STAGE;

CREATE OR REPLACE FILE FORMAT {{my_db}}.RAW.GOV_CSV
  TYPE = 'CSV'
  FIELD_OPTIONALLY_ENCLOSED_BY = '"'
  SKIP_HEADER = 1;


**What's in the roster:** six rows documenting exactly what you found in Sections 1–3 — which roles requested access to which objects, and whether that access was approved or denied. Notice the two `denied` rows for `CUSTOMERS` — that's the finding from your audit: the raw PII table was correctly locked down. This CSV is the formal record of that decision.

**Why stage and load it (rather than query SHOW GRANTS directly)?** Snowflake already knows who has what (`ACCOUNT_USAGE.GRANTS_TO_ROLES`). But the *review decision* — "was this grant approved by a human?" — lives outside Snowflake, in a ticketing system, a GRC tool, or a compliance spreadsheet. The load path (write → stage → COPY INTO) is how you bring that external judgment into Snowflake as a governed, queryable table alongside the grants it references.

In [ ]:
# PUT the access-review roster to your stage
csv_content = """role_name,object_name,privilege,last_reviewed,decision
ANALYST_RL,ROASTERY_INTELLIGENCE.V_FEEDBACK_SAFE,SELECT,2026-07-20,approved
ANALYST_RL,ROASTERY_INTELLIGENCE.FEEDBACK_SCORED_SUMMARY,SELECT,2026-07-20,approved
ANALYST_RL,ROASTERY_SALES.CUSTOMERS,SELECT,2026-07-20,denied
DE_RL,RAW.FEEDBACK_SOURCE,OWNERSHIP,2026-07-20,approved
DE_RL,ROASTERY_SALES.CUSTOMERS,SELECT,2026-07-20,denied
PLATPLUS_ADMIN_RL,ROASTERY_INTELLIGENCE.V_PRODUCTS_ENRICHED,SELECT,2026-07-20,approved
"""

tmp_path = '/tmp/access_review.csv'
with open(tmp_path, 'w') as f:
    f.write(csv_content)

session.file.put(tmp_path, f'@{my_db}.RAW.GOV_STAGE', auto_compress=False, overwrite=True)
print('File uploaded to stage.')


In [ ]:
-- Create the governed table and load it
CREATE OR REPLACE TABLE {{my_db}}.RAW.ACCESS_REVIEW (
  role_name     VARCHAR,
  object_name   VARCHAR,
  privilege     VARCHAR,
  last_reviewed DATE,
  decision      VARCHAR
);

COPY INTO {{my_db}}.RAW.ACCESS_REVIEW
FROM @{{my_db}}.RAW.GOV_STAGE/access_review.csv
FILE_FORMAT = (FORMAT_NAME = '{{my_db}}.RAW.GOV_CSV');


In [ ]:
-- Govern it: grant read to the analyst and confirm ownership
GRANT SELECT ON TABLE {{my_db}}.RAW.ACCESS_REVIEW TO ROLE {{my_analyst_role}};
SHOW GRANTS ON TABLE {{my_db}}.RAW.ACCESS_REVIEW;


> 🔎 **Review.** `COPY INTO` reports **6 rows loaded, 0 errors** — the file format parsed the header and dates cleanly. `SHOW GRANTS` shows exactly two rows: your `OWNERSHIP` and the `SELECT` you just granted the analyst. You didn't just *land* the data — you decided who could read it, at the moment it landed. That's the admin's job on the load path: control both *where* data lands and *who* can see it, from the start. (Because you own this table in your own database, `SHOW GRANTS ON TABLE` works — on a shared object you don't own, it wouldn't.)

---
## 5 · Govern marketplace access (Lab 06 — Marketplace)

You've fixed the internal leak and documented your audit. But governance isn't a one-time cleanup — it's an ongoing practice. Case in point: while you were fixing the PII issue, the analyst team filed a request for access to the Alpine Peaks book catalog (a Marketplace listing) for their book–coffee pairing work.

This is the reality of admin life: requests don't wait until your last fix is done. Each one is a governance decision — how much do you hand over, and through what mechanism? The wrong answer is "grant them everything and move on." The right answer is a controlled slice that you own and can revoke.

> 📌 **The `ALPINE_PEAKS_PUBLISHING` database is already here** — it was provisioned as part of the account setup. The cell below calls `ACCESS_ALPINE_PEAKS_PUBLISHING_DATA` to ensure your role has `IMPORTED PRIVILEGES` on the shared database (Lab 06 would have done this, but we include it here so the capstone is self-contained).

> 📌 **Recall Lab 06:** a Marketplace listing arrives as an imported, read-only database owned by `ACCOUNTADMIN`.

In [ ]:
USE ROLE {{my_role}};
USE WAREHOUSE {{my_wh}};

-- Ensure our role can access the shared marketplace database
-- (Lab 06 would have done this; we repeat it here so the capstone is self-contained)
CALL PLATPLUS_DB.RESOURCES.ACCESS_ALPINE_PEAKS_PUBLISHING_DATA('{{my_role}}');

In [ ]:
-- Inspect the marketplace listing you're being asked to share
SHOW DATABASES LIKE 'ALPINE_PEAKS_PUBLISHING';


The obvious move is to grant the analyst `IMPORTED PRIVILEGES` on the whole database. Try it — and watch what happens.

In [ ]:
-- EXPECTED to fail: a workspace admin cannot reshare a marketplace listing account-wide
GRANT IMPORTED PRIVILEGES ON DATABASE ALPINE_PEAKS_PUBLISHING TO ROLE {{my_analyst_role}};


> 🔒 **Blocked — and that's the lesson.** Resharing an entire Marketplace database account-wide is an `ACCOUNTADMIN` / `MANAGE GRANTS` action, not a workspace admin's to make. The governed pattern instead: publish a **curated projection** of exactly the columns you approve, in your own database, and grant the analyst *that*. You control the slice; you don't hand over the whole share.

In [ ]:
USE ROLE {{my_role}};
USE WAREHOUSE {{my_wh}};

-- Govern the right way: publish an approved projection, grant only that
CREATE OR REPLACE VIEW {{my_db}}.RAW.V_MARKETPLACE_APPROVED AS
SELECT title, category, release_date
FROM ALPINE_PEAKS_PUBLISHING.BOOKS.FORTHCOMING_RELEASES;

GRANT USAGE ON DATABASE {{my_db}}       TO ROLE {{my_analyst_role}};
GRANT USAGE ON SCHEMA {{my_db}}.RAW     TO ROLE {{my_analyst_role}};
GRANT SELECT ON VIEW {{my_db}}.RAW.V_MARKETPLACE_APPROVED TO ROLE {{my_analyst_role}};


In [ ]:
-- Confirm the analyst can consume the approved slice
USE ROLE {{my_analyst_role}};
USE WAREHOUSE PLATPLUS_WH;  -- (Uses PLATPLUS_WH because the analyst role doesn't have USAGE on your personal warehouse)
SELECT COUNT(*) AS approved_books, COUNT(DISTINCT category) AS categories
FROM {{my_db}}.RAW.V_MARKETPLACE_APPROVED;


In [ ]:
-- Prove the analyst got ONLY the approved slice, not the raw share (EXPECTED to fail)
SELECT COUNT(*) FROM ALPINE_PEAKS_PUBLISHING.BOOKS.FORTHCOMING_RELEASES;


> ✅ **The analyst reads 88 books across 8 categories through your view — but is denied on the raw share.** That's governed marketplace access: they consume exactly the columns you approved (`author` withheld), through an object you own and can revoke, without ever holding privileges on the imported database itself. You decided the slice; the org's Marketplace grant stayed locked down.

---
## 6 · Audit history & recover (Lab 08 — Time Travel)

You've now built several governed objects: a masked summary table, an access-review roster, a curated marketplace view. That's a governance layer worth protecting. But what protects *it*?

Admins fat-finger things — a wrong `DROP`, a bad `ALTER`. The governed objects you just built are exactly as vulnerable to human error as any other table. Time Travel is the safety net that lets you *prove* what an object looked like before a change and *undo* it — and critically, `UNDROP` restores the grants too, not just the data. Recovery without the access model intact is only half a recovery.

> 📌 **Recall Lab 08:** `UNDROP` recovers a dropped object from Time Travel; `AT/BEFORE` inspects a table's prior state.

In [ ]:
USE ROLE {{my_role}};
USE WAREHOUSE {{my_wh}};

-- A governed object worth protecting: a point-in-time copy of the access review
CREATE OR REPLACE TABLE {{my_db}}.RAW.GOVERNANCE_AUDIT AS
SELECT * FROM {{my_db}}.RAW.ACCESS_REVIEW;

GRANT SELECT ON TABLE {{my_db}}.RAW.GOVERNANCE_AUDIT TO ROLE {{my_analyst_role}};
SELECT COUNT(*) AS rows_before FROM {{my_db}}.RAW.GOVERNANCE_AUDIT;


In [ ]:
-- The accident: a governed table dropped by mistake
DROP TABLE {{my_db}}.RAW.GOVERNANCE_AUDIT;


> 🔎 **Prove it's gone first.** Before recovering, confirm the drop actually took effect — verify the failure state before you rely on the fix. The `SHOW` below returns **no rows**; a `SELECT * FROM … GOVERNANCE_AUDIT` would raise *"does not exist or not authorized."*

In [ ]:
-- Confirm the table is really gone (expect zero rows returned)
SHOW TABLES LIKE 'GOVERNANCE_AUDIT' IN SCHEMA {{my_db}}.RAW;


In [ ]:
-- Recover it from Time Travel
UNDROP TABLE {{my_db}}.RAW.GOVERNANCE_AUDIT;
SELECT COUNT(*) AS rows_after_recovery FROM {{my_db}}.RAW.GOVERNANCE_AUDIT;


In [ ]:
-- Did the grants survive the recovery, or does access have to be rebuilt?
SHOW GRANTS ON TABLE {{my_db}}.RAW.GOVERNANCE_AUDIT;


> ✅ **Recovered — 6 rows back, and the grants came with it.** `UNDROP` didn't just restore the data; `SHOW GRANTS` shows your `OWNERSHIP` and the analyst's `SELECT` intact. For an admin that's the whole point: recovery preserves the *access model*, not just the bytes — you don't have to rebuild governance after an accident. Time Travel turns a wrong `DROP` into a 10-second fix.

---
## 7 · Verify — prove the governance works (synthesis)

You've built governance across four surfaces in this lab:
- **Masking** (Section 3) — PII hidden from the analyst role
- **Load-path grants** (Section 4) — access decided at the moment data landed
- **Marketplace projection** (Section 5) — a controlled slice, not wholesale resharing
- **Recovery** (Section 6) — grants survive an accidental DROP

Now prove the most critical one actually works end-to-end: the masking policy that started this whole lab. You spotted the `MOST_RECENT_REVIEWER` leak in Section 2, fixed it in Section 3 — but you never *proved* it masks under the analyst role. That's what this section closes: the governance isn't real until you demonstrate it holds under a different identity.

In [ ]:
USE ROLE {{my_role}};
GRANT USAGE ON DATABASE {{my_db}}          TO ROLE {{my_analyst_role}};
GRANT USAGE ON SCHEMA {{my_db}}.RAW        TO ROLE {{my_analyst_role}};
GRANT SELECT ON TABLE {{my_db}}.RAW.FEEDBACK_SCORED_SUMMARY TO ROLE {{my_analyst_role}};


In [ ]:
USE ROLE {{my_analyst_role}};
USE WAREHOUSE PLATPLUS_WH;  -- (Uses PLATPLUS_WH because the analyst role doesn't have USAGE on your personal warehouse)
SELECT store_name, most_recent_reviewer FROM {{my_db}}.RAW.FEEDBACK_SCORED_SUMMARY;


> ✅ **`MOST_RECENT_REVIEWER` now reads `***MASKED***` for `{{my_analyst_role}}`.** Same table, same rows — the only thing that changed is which role is asking. Switch back and re-run to confirm `{{my_role}}` still sees the real value.

In [ ]:
USE ROLE {{my_role}};
USE WAREHOUSE {{my_wh}};
SELECT store_name, most_recent_reviewer FROM {{my_db}}.RAW.FEEDBACK_SCORED_SUMMARY;


### 🧭 Open reflection

Before you wrap up, answer these for yourself (or write them down — there's no single right answer):

- **What would you flag in a real review?** The masking fix is targeted and correct — but is there anything about *how* the leak got there in the first place that you'd raise with the DA team?
- **Section 1 vs. Section 2** — the raw `CUSTOMERS` table was already locked down, but the leak came through a derived summary. What does that tell you about where governance needs to apply?
- **What's still missing?** A resource monitor and a masking policy on one table is a start, not a finished governance program. What would you check next if this were a real production audit?
- **Which lab skill was hardest to recall?** Your signal for what to review before doing a real governance audit.
- **The `IMPORTED PRIVILEGES` wall in Section 5** — in a real org, would you have known the governed alternative (a curated view) without hitting that error first? What does that say about learning governance by *trying* vs by *being told*?

### Check your work
Run the next cell to **check your work**. It confirms the masking policy is attached and the resource monitor exists — no need to read the code.

> 📌 **Note:** This only checks sections that create persistent governance objects (masking policies, resource monitors). Sections that produce one-off query results (audit queries, access-denied demos, UNDROP exercises) aren't checked here — you verified those by reading their output as you went.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
user = session.get_current_user().strip('"')
my_db = user + '_DB'
mon_name = user + '_RM'

try:
    rows = session.sql(f"""
        SELECT POLICY_NAME FROM TABLE({my_db}.INFORMATION_SCHEMA.POLICY_REFERENCES(
          REF_ENTITY_NAME => '{my_db}.RAW.FEEDBACK_SCORED_SUMMARY',
          REF_ENTITY_DOMAIN => 'TABLE'
        )) WHERE REF_COLUMN_NAME = 'MOST_RECENT_REVIEWER'
    """).collect()
    mask_ok = len(rows) >= 1
except Exception:
    mask_ok = False

try:
    mon_rows = session.sql(f"SHOW RESOURCE MONITORS LIKE '{mon_name}'").collect()
    mon_ok = len(mon_rows) >= 1
except Exception:
    mon_ok = False

try:
    access_rows = session.sql(f"SELECT COUNT(*) AS cnt FROM {my_db}.RAW.ACCESS_REVIEW").collect()
    load_ok = len(access_rows) > 0 and access_rows[0]['CNT'] >= 1
except Exception:
    load_ok = False

try:
    audit_rows = session.sql(f"SELECT COUNT(*) AS cnt FROM {my_db}.RAW.GOVERNANCE_AUDIT").collect()
    tt_ok = len(audit_rows) > 0 and audit_rows[0]['CNT'] >= 1
except Exception:
    tt_ok = False

print(f"{'✅' if mask_ok else '⛔'}  Masking policy attached to MOST_RECENT_REVIEWER")
print(f"{'✅' if mon_ok else '⛔'}  Resource monitor {mon_name} exists")
print(f"{'✅' if load_ok else '⛔'}  Load path: ACCESS_REVIEW table populated")
print(f"{'✅' if tt_ok else '⛔'}  Time Travel: GOVERNANCE_AUDIT table recovered")
print('-' * 52)
if mask_ok and mon_ok and load_ok and tt_ok:
    print('🏁  Roastery Intelligence — Governance layer — COMPLETE')
else:
    print('⚠️  Not all checks passed yet — re-run the relevant cell above, then run this cell again.')

### Labs recalled in this capstone

| Lab | Skill | Where it appeared |
|-----|-------|-------------------|
| 01 | Navigate objects | Section 1 — object map of the investigation schemas |
| 02 | Verify results | Section 1 — corroborate the no-access finding |
| 03 | RBAC & access | Sections 3 & 7 — masking policy + role-scoped grants |
| 04 | Credits & cost | Sections 1 & 3 — query cost audit + resource monitor |
| 05 | AI functions | Section 2 — consuming the AI-scored summary handoff |
| 06 | Marketplace | Section 5 — govern access via an approved projection |
| 07 | Getting data in | Section 4 — govern the load path (stage + COPY) |
| 08 | Time Travel | Section 6 — UNDROP recovery with grants intact |
| 09 | Unstructured AI | (referenced) — the coffee-KB pipeline the DE built |

---
### ❄️ Key Takeaways
- Auditing an investigation is a separate job from validating its conclusion — check cost and access, not just correctness
- A locked-down raw table doesn't guarantee a locked-down platform — PII can resurface in a derived or summary table built downstream
- Targeted remediation (a masking policy on one column) is almost always the right scope — not a blanket policy on the whole table
- `INFORMATION_SCHEMA.POLICY_REFERENCES` lets you verify a masking policy is attached without switching roles first
- Resource monitors and masking policies are both `ACCOUNTADMIN`-gated actions in this account — wrapped in owner's-rights procs so you can call them directly
- Governance work is never "done" after one fix — a real review asks what's still missing, not just whether this one leak is closed
- Map your governance surface first (`INFORMATION_SCHEMA`) — you can't govern objects you haven't inventoried
- A workspace admin can't reshare a Marketplace listing account-wide — the governed move is a curated projection you own and can revoke
- `UNDROP` recovers a dropped object *with its grants* — recovery restores the access model, not just the data
- Govern data at the moment it lands (grant on load), not as an afterthought

**The Other Personas:**
- The **Data Analyst** notebook builds the reusable dip-detection view + chart you audited the cost of in Section 1
- The **Data Engineer** notebook builds the pipeline behind `V_FEEDBACK_SAFE` — the object you'd expect a well-governed feedback pipeline to publish

### Notebook completion. 🏁
You're done — your notebook is saved automatically.

> 💡 **Moving on to another lab?** Shut down this notebook's **kernel** — click **Connected** at the top → **Shut down kernel**. All your lab notebooks share one compute service, so clearing a finished kernel frees up memory for the next lab. Your service stays running, ready in seconds.

> 💡 **Finished for the day?** **Suspend your service** — click **Connected** → hover your service name → **Suspend**. This stops compute; it resumes automatically next time you run a cell.